# 04 · Robustness checks

**Goal:** stress-test the findings from notebooks 00-03 and answer their open questions. Nothing here is new modelling; it is about deciding how much to trust what we already saw.

**How it fits the project:** notebook 03 showed that the segment-wise split (leaky) (shuffle 4 s windows, ignore which participant they came from) scores far higher than a subject-wise split. Before believing that story we should check the obvious objections: Was the data pre-filtered? Was the subject-wise per-subject AUC of 0.70 a lucky split? Does the result just come from recording length? Are the groups separated by artifacts (EOG, EMG) rather than brain rhythms? Each section below checks one objection.

**How to read it:** every code cell has a markdown cell above it with **Question**, **How** and **What to look for**. This notebook is *not run here* — run it yourself (conda env `medimg`, CPU only). Sections 2-5 reload all 121 recordings, and sections 2-3 fit random forests, so expect a few minutes per section.

**Sections**
1. Is the data pre-filtered, or is 50 Hz just mains noise?
2. Is the subject-wise per-subject AUC stable, and does it beat recording length?
3. What happens if every participant is cropped to the same length?
4. Do EOG and EMG artifacts differ between the groups?
5. Do the group-average spectra differ (30 Hz bump, alpha map)?

## Section 1 — Is the data pre-filtered, or is 50 Hz just mains noise?

**Question:** Notebook 00 only plotted up to 50 Hz, so we could not tell whether the signal is flat above 50 Hz (already low-pass filtered by whoever recorded it) or whether 50 Hz is just a narrow power-line spike. Which is it?

**How:**
- Load the same participant as notebook 00 (`v10p`), **without** any filtering.
- Compute a **Welch PSD** (cut the signal into 4 s segments, Fourier-transform each, average them for a smoother estimate) up to the **Nyquist frequency** = half the sampling rate = 128 / 2 = **64 Hz**. Nothing can exist above Nyquist, so this is the whole picture.
- Convert power to dB (`10*log10`) and plot one thin line per channel. A dashed line marks 50 Hz; a dotted line marks 40 Hz (our feature cut-off).

**What to look for:** if the curve keeps sliding smoothly downward past 50 Hz all the way to 64 Hz, with only a thin spike sitting exactly on 50 Hz, the data is **not** pre-filtered and 50 Hz is just **mains noise** (Europe/Asia use 50 Hz mains). If instead the power drops to a flat floor right after an edge (50 or 60 Hz) and stays there, the recording was **low-pass filtered** before we received it. This settles the "likely pre-filtered" guess from notebook 00.

In [ ]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from mne.time_frequency import psd_array_welch
from src.load import list_subjects, load_subject
from src.channels import CHANNELS, SFREQ

# same participant checked in notebook 00 (first ADHD file)
path = next(c["path"] for c in list_subjects() if c["subject_id"] == "v10p")
x = load_subject(path)                        # 19 x samples, in uV, NOT filtered

# Welch PSD all the way up to Nyquist (128 / 2 = 64 Hz)
psd, freqs = psd_array_welch(x, SFREQ, fmin=1, fmax=64, n_fft=512, verbose=False)
psd_db = 10 * np.log10(psd)                   # to dB, like MNE's PSD plot

plt.figure(figsize=(10, 4))
plt.plot(freqs, psd_db.T, lw=0.5)             # one thin line per channel
plt.axvline(50, color="k", ls="--", lw=1, label="50 Hz (mains?)")
plt.axvline(40, color="grey", ls=":", lw=1, label="40 Hz (feature cut-off)")
plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (dB re 1 uV^2/Hz)")
plt.title("v10p raw PSD up to Nyquist (64 Hz)"); plt.legend(); plt.tight_layout()
plt.show()

## Section 2 — Is the subject-wise per-subject AUC stable, and does it beat length?

**Question (build step):** First we need the window-level feature table from notebook 03. Can we rebuild it here?

**How:** for every participant, `window_features` applies the 1-40 Hz filter, cuts the recording into 4 s windows, and turns each window into the 76 band-power features. We stack all participants into one table `win` and split it into `X` (features), `y` (labels) and `g` (subject id, the grouping key that says which windows belong together).

**What to look for:** the shape should match notebook 03 (4173 windows x 78 columns).

In [ ]:
import pandas as pd
from src.features import window_features

subjects = pd.DataFrame(list_subjects())
parts = []
for _, k in subjects.iterrows():
    f = window_features(load_subject(k.path))   # 1-40 Hz filter + 4 s windows (as in notebook 03)
    f["subject_id"], f["label"] = k.subject_id, k.label
    parts.append(f)
win = pd.concat(parts, ignore_index=True)

X, y, g = win.drop(columns=["subject_id", "label"]), win.label, win.subject_id
print(win.shape)                                # expect (4173, 78), same as notebook 03

**Question:** Notebook 03 got a subject-wise per-subject AUC of 0.70, but from a **single** random split. Was that luck? And does it really beat the recording-duration baseline (0.65 in notebook 01)?

**How:**
- Repeat the subject-wise evaluation for **10 different split seeds**. For each seed, `StratifiedGroupKFold(5, shuffle=True, random_state=seed)` keeps all of a participant's windows together in one fold (no participant in both train and test) while keeping the ADHD/Control mix even. The random forest is exactly the one from notebook 03 (300 trees, fixed `random_state=42`), so only the *split* changes between runs.
- `cross_val_predict` gives each window an out-of-fold ADHD probability; we **average the windows of each participant** into one score, then take the AUC over the 121 participants.
- On the **same 10 seeds**, run the recording-duration baseline logistic regression at subject level (one row per participant, so a plain `StratifiedKFold` is already a by-subject split).
- Print mean +/- standard deviation for both.

**What to look for:** compare the two mean AUCs and their spreads. If the EEG interval (mean +/- std) sits clearly above the recording-duration interval, EEG adds real information beyond recording length. If they overlap, we **cannot** claim EEG beats the simple recording-duration shortcut. The std also tells us how lucky or unlucky the single 0.70 was. *(This cell fits 50 random forests twice over; it is the slowest in the notebook.)*

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (StratifiedGroupKFold, StratifiedKFold,
                                      cross_val_predict, cross_val_score)
from sklearn.metrics import roc_auc_score

model = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)  # same as notebook 03
seeds = range(10)

# --- EEG random forest, subject-wise split, averaged to one score per subject ---
eeg_auc = []
for s in seeds:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=s)
    p = cross_val_predict(model, X, y, groups=g, cv=cv, method="predict_proba")[:, 1]
    per_subject = pd.DataFrame({"subject_id": g, "label": y, "p": p}).groupby("subject_id").mean()
    eeg_auc.append(roc_auc_score(per_subject.label, per_subject.p))

# --- recording-duration baseline, one row per subject, same seeds ---
subjects["seconds"] = [load_subject(pth).shape[1] / SFREQ for pth in subjects.path]
len_auc = []
for s in seeds:
    cv = StratifiedKFold(5, shuffle=True, random_state=s)
    a = cross_val_score(LogisticRegression(), subjects[["seconds"]], subjects.label,
                        cv=cv, scoring="roc_auc")
    len_auc.append(a.mean())

print(f"EEG per-subject AUC (subject-wise)   : {np.mean(eeg_auc):.3f} +/- {np.std(eeg_auc):.3f}")
print(f"Recording-duration baseline (subject): {np.mean(len_auc):.3f} +/- {np.std(len_auc):.3f}")

## Section 3 — What if every participant is cropped to the same length?

**Question (build step):** Recording length differs between groups (notebook 01) and longer recordings give more windows (notebook 03: ADHD 2330 windows vs Control 1843). Both are shortcuts that have nothing to do with brain activity. If we crop every participant to the first 60 s, each participant gives the same number of windows — can we build that table?

**How:** the shortest recording is 62.4 s, so 60 s fits everyone. We call `window_features(..., max_seconds=60)` (the new argument added to `src/features.py`): it keeps only the first 60 s *before* windowing, giving 60 / 4 = **15 windows per participant**.

**What to look for:** shape should be (1815, 78) = 121 x 15, and every participant should have exactly 15 windows. Classes are now nearly balanced (61 vs 60 participants -> 915 vs 900 windows), so the "always say ADHD" accuracy baseline drops from 0.56 to about 0.50.

In [ ]:
parts60 = []
for _, k in subjects.iterrows():
    f = window_features(load_subject(k.path), max_seconds=60)   # first 60 s -> exactly 15 windows
    f["subject_id"], f["label"] = k.subject_id, k.label
    parts60.append(f)
win60 = pd.concat(parts60, ignore_index=True)

X60, y60, g60 = win60.drop(columns=["subject_id", "label"]), win60.label, win60.subject_id
print(win60.shape)                                   # expect (1815, 78)
print(win60.groupby("subject_id").size().value_counts())   # should be 15 for every subject

**Question:** With length and window-count made equal, does the segment-wise-vs-subject-wise gap shrink, and does the subject-wise per-subject AUC change?

**How:**
- Rerun the **segment-wise split (leaky)** (`StratifiedKFold`, windows shuffled) vs the **subject-wise split** (`StratifiedGroupKFold`, split by participant) at the window level on the 60 s data, reporting accuracy and AUC.
- Repeat the 10-seed subject-wise per-subject AUC from section 2, now on the cropped data.

**What to look for:** the segment-wise-vs-subject-wise gap should *still* be large — leakage comes from the per-participant "fingerprint", not from recording length — which would show the gap is real and not just a length effect. Then compare the cropped per-subject AUC with the full-length one from section 2: if it falls toward 0.5, much of the earlier signal was really recording length; if it holds up, there is some length-independent EEG signal.

In [ ]:
from sklearn.model_selection import cross_validate

# segment-wise (leaky) vs subject-wise, window-level (same as notebook 03 but on 60 s crops)
for name, cv in {"segment-wise (leaky)": StratifiedKFold(5, shuffle=True, random_state=42),
                 "subject-wise": StratifiedGroupKFold(5, shuffle=True, random_state=42)}.items():
    r = cross_validate(model, X60, y60, groups=g60, cv=cv, scoring=["accuracy", "roc_auc"])
    print(f"{name:18s} accuracy {r['test_accuracy'].mean():.2f}   AUC {r['test_roc_auc'].mean():.2f}")

# repeated subject-wise per-subject AUC on the cropped data (same recipe as section 2)
eeg_auc60 = []
for s in seeds:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=s)
    p = cross_val_predict(model, X60, y60, groups=g60, cv=cv, method="predict_proba")[:, 1]
    per_subject = pd.DataFrame({"subject_id": g60, "label": y60, "p": p}).groupby("subject_id").mean()
    eeg_auc60.append(roc_auc_score(per_subject.label, per_subject.p))
print(f"\nEEG per-subject AUC, 60 s equal length: {np.mean(eeg_auc60):.3f} +/- {np.std(eeg_auc60):.3f}")

## Section 4 — Do EOG and EMG artifacts differ between groups?

**Question (build step):** Notebook 00 saw big frontal swings (likely EOG) and high 17-40 Hz power at T8 (likely EMG) in one participant. If such artifacts are more common in one group, a classifier could "diagnose" by picking them up instead of brain activity. Do they differ across all 121 participants?

**How:** `artifact_metrics` filters each participant to 1-40 Hz, cuts 4 s windows, and returns one number per participant for each metric:
- **EOG** = the share of windows where Fp1 **or** Fp2 has a **peak-to-peak** swing (max minus min within the window) over **150 uV**. Large frontal swings are the signature of blinks and eye movements.
- **EMG** = **relative 20-40 Hz power** (that band's share of total 1-40 Hz power) at the four edge channels **T7, T8, F7, F8**, averaged over windows. EMG activity adds broad high-frequency power, strongest near the jaw and temples.

**What to look for:** the printed per-group medians give a first hint of any difference (we test it properly in the next cell).

In [ ]:
import mne

def artifact_metrics(x, win_s=4):
    """Per-subject artifact summary from 1-40 Hz filtered data (one number per metric)."""
    x = mne.filter.filter_data(x, SFREQ, 1.0, 40.0, verbose=False)
    n = SFREQ * win_s
    n_win = x.shape[1] // n
    w = x[:, :n_win * n].reshape(x.shape[0], n_win, n).transpose(1, 0, 2)   # (windows, channels, samples)
    idx = {ch: i for i, ch in enumerate(CHANNELS)}

    # EOG: fraction of windows where Fp1 OR Fp2 swings more than 150 uV peak-to-peak
    pp = w.max(-1) - w.min(-1)                                    # (windows, channels)
    eog = ((pp[:, idx["Fp1"]] > 150) | (pp[:, idx["Fp2"]] > 150)).mean()

    # EMG: relative 20-40 Hz power (share of 1-40 Hz power), averaged over windows
    psd, freqs = psd_array_welch(w, SFREQ, fmin=1, fmax=40, n_fft=n, verbose=False)
    hi = (freqs >= 20) & (freqs < 40)
    rel = psd[..., hi].sum(-1) / psd.sum(-1)                      # (windows, channels)

    out = {"eog_share": eog}
    for ch in ["T7", "T8", "F7", "F8"]:
        out[f"emg_{ch}"] = rel[:, idx[ch]].mean()
    return out

rows = []
for _, k in subjects.iterrows():
    m = artifact_metrics(load_subject(k.path))
    m["group"], m["label"] = k.group, k.label
    rows.append(m)
art = pd.DataFrame(rows)
print(art.groupby("group").median().round(3))

**Question:** Are these artifact levels actually different between ADHD and Control, or could the difference be chance?

**How:** draw box plots per group (box = middle 50% of participants, line = median) for the EOG metric and the four EMG channels, then run a **Mann-Whitney U test** for each (a rank test that does not assume a bell curve) and print the two group medians with the p-value.

**What to look for:** a small p (say < 0.05) together with clearly separated boxes means that artifact really differs between groups — a worry, because the model could use it instead of brain activity. Large p-values (like TBR in notebook 02) mean the groups look similar on that artifact, which is reassuring. We run 5 tests here, so treat a single borderline p with caution.

In [ ]:
from scipy.stats import mannwhitneyu

art.boxplot(column="eog_share", by="group")
art.boxplot(column=["emg_T7", "emg_T8", "emg_F7", "emg_F8"], by="group", figsize=(9, 4))
plt.show()

print("metric       ADHD    Control   p (Mann-Whitney)")
for col in ["eog_share", "emg_T7", "emg_T8", "emg_F7", "emg_F8"]:
    a = art[col][art.label == 1]
    c = art[col][art.label == 0]
    p = mannwhitneyu(a, c).pvalue
    print(f"{col:10s} {a.median():7.3f} {c.median():8.3f}   {p:.3f}")

## Section 5 — Do the group-average spectra differ?

**Question (build step):** To compare the *shape* of the spectrum between groups we need the PSD of every participant. Can we compute it?

**How:** filter each participant to 1-40 Hz, then take a Welch PSD from 1 to 45 Hz. Stack into an array of shape (participants, channels, frequencies) and keep each participant's label.

**What to look for:** shape should be (121, 19, number-of-frequency-bins). Note: the 1-40 Hz filter means power between 40 and 45 Hz is already rolling off — that tail of the curve is the filter, not the brain.

In [ ]:
psd_all, labels = [], []
for _, k in subjects.iterrows():
    xf = mne.filter.filter_data(load_subject(k.path), SFREQ, 1.0, 40.0, verbose=False)
    psd, freqs = psd_array_welch(xf, SFREQ, fmin=1, fmax=45, n_fft=512, verbose=False)
    psd_all.append(psd)                       # (channels, freqs)
    labels.append(k.label)
psd_all = np.array(psd_all)                   # (subjects, channels, freqs)
labels = np.array(labels)
print(psd_all.shape, "freqs:", round(freqs[0], 2), "to", round(freqs[-1], 2), "Hz")

**Question:** Notebook 00 saw a small bump near 30 Hz in one participant. Does it appear across many participants, and does the spectrum differ between groups?

**How:** average the **log** PSD (in dB) across participants within each group, separately for five channels: Cz (centre), O1/O2 (back of head), Fp1 (front, EOG-prone) and T8 (EMG-prone). Averaging in dB (a geometric mean) stops one very loud participant from dominating. Plot ADHD vs Control per channel, with a reference line at 30 Hz.

**What to look for:** a real 30 Hz rhythm would show as a small bump in the average of *both* groups; if it disappears after averaging, it was probably one participant or noise. Where the two group curves separate (which channels, which bands) is where any group difference lives. Watch O1/O2 for the alpha (~10 Hz) peak and T8/Fp1 for broadband high-frequency power (a sign of artifacts).

In [ ]:
idx = {ch: i for i, ch in enumerate(CHANNELS)}
db = 10 * np.log10(psd_all)                   # (subjects, channels, freqs)

show = ["Cz", "O1", "O2", "Fp1", "T8"]
fig, axes = plt.subplots(len(show), 1, figsize=(8, 11), sharex=True)
for ax, ch in zip(axes, show):
    ax.plot(freqs, db[labels == 1, idx[ch]].mean(0), label="ADHD")
    ax.plot(freqs, db[labels == 0, idx[ch]].mean(0), label="Control")
    ax.axvline(30, color="grey", ls=":", lw=1)     # the ~30 Hz bump seen in notebook 00
    ax.set_title(ch); ax.set_ylabel("dB"); ax.legend(loc="upper right")
axes[-1].set_xlabel("Frequency (Hz)")
plt.tight_layout(); plt.show()

**Question:** Notebook 00 found alpha strongest over the centre of the head (C4/C3/Cz) for one participant. Does that topography hold across participants, and does relative alpha differ between groups?

**How:** from the stored PSDs, compute **relative alpha** per channel = alpha power (8-13 Hz) divided by total power (1-40 Hz) — the same definition as `band_features` in notebook 02. Average over participants in each group and print a table sorted by the ADHD value, with the ADHD-minus-Control difference.

**What to look for:** which channels sit at the top — central channels (Cz, C3, C4) would confirm the notebook-00 pattern (likely mu rhythm) as a group feature, not just one participant. The difference column shows whether either group has systematically more alpha anywhere; small differences fit the weak EEG separation we saw in notebook 02.

In [ ]:
alpha = (freqs >= 8) & (freqs < 13)
total = (freqs >= 1) & (freqs < 40)                  # same 1-40 Hz total as band_features
rel_alpha = psd_all[..., alpha].sum(-1) / psd_all[..., total].sum(-1)   # (subjects, channels)

tab = pd.DataFrame(rel_alpha, columns=CHANNELS)
tab["label"] = labels
means = tab.groupby("label").mean().T                # channels x [Control, ADHD]
means.columns = ["Control", "ADHD"]
means["ADHD - Control"] = means["ADHD"] - means["Control"]
print(means.sort_values("ADHD", ascending=False).round(3))

## Key takeaways

*Fill in after running.*

- Section 1 (pre-filtering):
- Section 2 (stability vs length):
- Section 3 (equal length):
- Section 4 (artifacts):
- Section 5 (spectra / alpha map): 

## Open questions

*Fill in after running.*

- 